
# Entrenamiento de una combinación de hiperparámetros

**Autor:** Juan Carlos Alfaro Jiménez

Esta libreta **no contiene ninguna llamada a `MLflow`**. Su única responsabilidad es aislar el entrenamiento de un único `PipelineModel` de `MLlib` utilizando los hiperparámetros recibidos. Durante su ejecución independiente, calculará las métricas de evaluación, generará las figuras de diagnóstico y guardará todos los artefactos físicos resultantes directamente en un volumen de `Unity Catalog`.

### ¿Por qué desacoplar el entrenamiento de `MLflow`?

`Spark Connect` gestiona los modelos entrenados por `MLlib` almacenándolos en una caché del lado del servidor, la cual tiene un límite estricto de **1`GB` por sesión**. Si intentásemos entrenar iterativamente múltiples combinaciones de hiperparámetros (un *grid search* tradicional) dentro de una misma sesión, estos modelos se acumularían rápidamente en memoria, provocando el fallo `ML_CACHE_SIZE_OVERFLOW_EXCEPTION`.

La solución arquitectónica óptima consiste en delegar el entrenamiento a esta libreta secundaria, invocándola de forma aislada mediante la instrucción `dbutils.notebook.run()` desde nuestra libreta orquestadora principal (`07_MLflow_Experimentation.ipynb`). 

De este modo, cada invocación levanta una **sesión de `Spark Connect` completamente limpia y nueva**, garantizando que la caché de memoria se libere correctamente al terminar cada ciclo de entrenamiento.


## 1. Importaciones y carga de utilidades compartidas

El *script* `07_Utils.py` contiene la configuración estructural del proyecto, la carga centralizada del conjunto de datos y la definición de los *splits* temporales. Al ejecutarlo, heredamos automáticamente todas sus variables y funciones en este espacio de nombres, garantizando la consistencia del entorno.

A continuación, importamos las librerías necesarias **exclusivamente** para la ejecución de esta libreta aislada:

In [0]:
exec(open("07_Utils.py").read(), globals())

In [0]:
import json
import gc
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
matplotlib.use("Agg")  # Non-interactive backend: safe on cluster drivers with no display

import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve
)

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    SQLTransformer,
    StandardScaler,
    StringIndexer,
    VarianceThresholdSelector,
    VarianceThresholdSelectorModel,
    VectorAssembler
)


## 2. Recepción de hiperparámetros

Los *widgets* son el mecanismo estándar en `Databricks` para parametrizar y pasar información dinámicamente entre libretas. En nuestro flujo, la libreta orquestadora inyecta todos estos valores simultáneamente utilizando el parámetro `arguments` de la instrucción `dbutils.notebook.run()`.

> **Importante sobre el tipado**: Todos los valores recibidos a través de los *widgets* llegan **siempre como cadenas de texto** (`String`). Es imprescindible convertirlos explícitamente a su tipo de dato correspondiente (flotantes, enteros o booleanos) antes de pasarlos al modelo o al *pipeline*.


### 2.1. Hiperparámetros del preprocesado

Estos hiperparámetros controlan el comportamiento de las etapas de transformación de datos previas al clasificador. No se definen aquí, esta libreta solo los recibe y los aplica:

* **`imputer_strategy`**: Estrategia de imputación para columnas numéricas con nulos (`median` es más robusta ante valores atípicos que `mean`).
* **`var_selector_threshold`**: Varianza mínima que debe tener una característica para no ser descartada.
* **`scaler_with_mean`**: Centrar los datos en la media antes de escalar (mantenido en `False` para no destruir la dispersidad del *one-hot encoding*).
* **`scaler_with_std`**: Normalizar la distribución a varianza unitaria.
* **`ohe_drop_last`**: Eliminar la última categoría en la codificación *one-hot* para evitar multicolinealidad perfecta.
* **`si_handle_invalid`**: Política del `StringIndexer` ante categorías nuevas (`keep` asigna un índice especial).
* **`si_order_type`**: Criterio de ordenación de categorías (`frequencyDesc` asigna el índice 0 a la más frecuente).
* **`ohe_handle_invalid`**: Política del codificador *one-hot* ante índices desconocidos en inferencia.
* **`asm_handle_invalid`**: Política del `VectorAssembler` ante nulos residuales (`error` actúa como barrera de calidad estricta).

In [0]:
# Default values are used only in interactive runs; notebook.run() overrides them
dbutils.widgets.text("imputer_strategy", "median")
dbutils.widgets.text("var_selector_threshold", "0.01")
dbutils.widgets.text("scaler_with_mean", "False")
dbutils.widgets.text("scaler_with_std", "True")
dbutils.widgets.text("ohe_drop_last", "True")
dbutils.widgets.text("si_handle_invalid", "keep")
dbutils.widgets.text("si_order_type", "frequencyDesc")
dbutils.widgets.text("ohe_handle_invalid", "keep")
dbutils.widgets.text("asm_handle_invalid", "error")

imputer_strategy = dbutils.widgets.get("imputer_strategy")
var_selector_threshold = float(dbutils.widgets.get("var_selector_threshold"))
scaler_with_mean = dbutils.widgets.get("scaler_with_mean").lower() == "true"
scaler_with_std = dbutils.widgets.get("scaler_with_std").lower() == "true"
ohe_drop_last = dbutils.widgets.get("ohe_drop_last").lower() == "true"
si_handle_invalid = dbutils.widgets.get("si_handle_invalid")
si_order_type = dbutils.widgets.get("si_order_type")
ohe_handle_invalid = dbutils.widgets.get("ohe_handle_invalid")
asm_handle_invalid = dbutils.widgets.get("asm_handle_invalid")

print(f"Imputer strategy: {imputer_strategy}")
print(f"Variance selector threshold: {var_selector_threshold}")
print(f"Scaler with mean: {scaler_with_mean}")
print(f"Scaler with standard deviation: {scaler_with_std}")
print(f"One-hot encoding drop last category: {ohe_drop_last}")
print(f"String indexer handle invalid: {si_handle_invalid}")
print(f"String indexer order type: {si_order_type}")
print(f"One-hot encoding handle invalid: {ohe_handle_invalid}")
print(f"Assembler handle invalid: {asm_handle_invalid}")


### 2.2. Hiperparámetros del clasificador

Estos hiperparámetros controlan directamente el comportamiento del algoritmo final (`LogisticRegression`):

* **`reg_param`**: Intensidad global de la penalización de regularización (`L2` puro o mezcla de `L1` y `L2`).
* **`elastic_net_param`**: Balance de la penalización `Elastic Net` (mezcla entre `Ridge` `0.0` y `Lasso` `1.0`).
* **`max_iter`**: Número máximo de iteraciones permitidas para el optimizador `L-BFGS`.
* **`family`**: Tipo de modelo (`binomial` especifica regresión logística binaria).
* **`standardization`**: Estandarización interna del clasificador (fijado a `False` porque el *pipeline* ya incluye un `StandardScaler`).
* **`threshold`**: Umbral de decisión por defecto.

In [0]:
dbutils.widgets.text("reg_param", "0.01")
dbutils.widgets.text("elastic_net_param", "0.0")
dbutils.widgets.text("max_iter", "100")
dbutils.widgets.text("family", "binomial")
dbutils.widgets.text("standardization", "False")
dbutils.widgets.text("threshold", "0.5")

reg_param = float(dbutils.widgets.get("reg_param"))
elastic_net_param = float(dbutils.widgets.get("elastic_net_param"))
max_iter = int(dbutils.widgets.get("max_iter"))
family = dbutils.widgets.get("family")
standardization = dbutils.widgets.get("standardization").lower() == "true"
threshold = float(dbutils.widgets.get("threshold"))

# Run tag: Unique visual identifier for this grid point
run_tag = f"lr__rp{reg_param}__en{elastic_net_param}__seed{seed}"

print(f"Regularization hyperparameter: {reg_param}")
print(f"Elastic net hyperparameter: {elastic_net_param}")
print(f"Maximum iterations: {max_iter}")
print(f"Family: {family}")
print(f"Standardization: {standardization}")
print(f"Threshold: {threshold}")
print(f"Run tag: {run_tag}")


## 3. Evaluadores, métricas y funciones auxiliares


### 3.1. Configuración general y visual

Primero, definimos los nombres de las columnas que generará el modelo y establecemos límites de seguridad, como `to_pandas_max_rows`, para evitar errores de memoria al generar gráficos. También centralizamos las dimensiones y la paleta de colores para asegurar consistencia visual en todas las figuras.

In [0]:
raw_prediction_column = "rawPrediction"
prediction_column = "prediction"
probability_column = "probability"
prob_fraud_column = "prob_fraud"

to_pandas_max_rows = 200_000

# Threshold sweep grid: shared between the threshold visualization and the
# optimal-threshold search loop so the logged value matches the figure
threshold_sweep_start = 0.01
threshold_sweep_stop = 0.99
threshold_sweep_steps = 99

fig_size_standard = (6, 5)
fig_size_wide = (8, 5)
fig_size_confusion = (11, 4)
fig_size_coef_width = 9
fig_size_coef_row_h = 0.30
fig_size_coef_min_h = 4

color_roc = "#1f77b4"
color_pr = "#ff7f0e"
color_calibration = "#2ca02c"
color_positive_coef = "#d62728"
color_negative_coef = "#1f77b4"
color_random_baseline = "gray"

calibration_n_bins = 10
coefficients_top_n = 30


### 3.2. Instanciación de evaluadores nativos

Los evaluadores de `MLlib` se dividen en dos grupos:

* **`BinaryClassificationEvaluator`**: Calcula métricas **independientes del umbral** (*AUC-ROC* y *AUC-PR*) integrando sobre todos los posibles umbrales de decisión. Son ideales para comparar el poder discriminativo de los modelos antes de establecer un umbral de corte definitivo.
* **`MulticlassClassificationEvaluator`**: Calcula métricas **dependientes del umbral** (*F1-score*, *precision*, *recall*, *accuracy*) usando el umbral de decisión por defecto del modelo (habitualmente `0.5`).

In [0]:
eval_auc_roc = BinaryClassificationEvaluator(
    labelCol = label_column,
    rawPredictionCol = raw_prediction_column,
    metricName = "areaUnderROC"
)
eval_auc_pr = BinaryClassificationEvaluator(
    labelCol = label_column,
    rawPredictionCol = raw_prediction_column,
    metricName = "areaUnderPR"
)

eval_f1 = MulticlassClassificationEvaluator(
    labelCol = label_column,
    predictionCol = prediction_column,
    metricName = "f1"
)
eval_precision = MulticlassClassificationEvaluator(
    labelCol = label_column,
    predictionCol = prediction_column,
    metricName = "weightedPrecision"
)
eval_recall = MulticlassClassificationEvaluator(
    labelCol = label_column,
    predictionCol = prediction_column,
    metricName = "weightedRecall"
)
eval_accuracy = MulticlassClassificationEvaluator(
    labelCol = label_column,
    predictionCol = prediction_column,
    metricName = "accuracy"
)


### 3.3. Función de cálculo de métricas

La función `compute_metrics` centraliza la llamada a los seis evaluadores definidos anteriormente. Esto evita la duplicación de código en las celdas posteriores al evaluar tanto el conjunto de entrenamiento como el de validación.

In [0]:
def compute_metrics(predictions):
    """
    Compute all six evaluation metrics.

    Returns a plain dictionary so values can be returned seamlessly
    to the orchestrator notebook without any complex transformations.
    """
    return {
        "auc_roc": eval_auc_roc.evaluate(predictions),
        "auc_pr": eval_auc_pr.evaluate(predictions),
        "f1": eval_f1.evaluate(predictions),
        "precision": eval_precision.evaluate(predictions),
        "recall": eval_recall.evaluate(predictions),
        "accuracy": eval_accuracy.evaluate(predictions)
    }


### 3.4. Función de transformación segura a `pandas`

La función `to_pandas_predictions` extrae la probabilidad de fraude (`prob_fraud`) del vector de probabilidades nativo de `Spark`, convirtiéndola en un valor escalar estándar. Como medida de seguridad crítica, esta función aplica un límite estricto de filas antes de materializar los datos en `pandas`, previniendo así posibles errores de desbordamiento de memoria en el nodo *driver* del clúster.

In [0]:
def to_pandas_predictions(predictions):
    """
    Convert the prediction columns.

    Extracts the fraud probability from the probability vector at index 1
    and adds it as a plain float column. Limits rows to prevent out-of-memory
    errors on large datasets.
    """
    return (
        predictions
        .select(label_column, probability_column, prediction_column)
        .limit(to_pandas_max_rows)
        .toPandas()
        .assign(
            **{
                prob_fraud_column: lambda df: df[probability_column].apply(
                    lambda values: float(values[1])
            )}
        )
    )


## 4. Funciones de generación de figuras

Cada función está diseñada para ser completamente pura respecto a la visualización: recibe únicamente los datos necesarios (`DataFrame` de `pandas` o `array` de `numpy`) y devuelve un objeto `Figure` de `matplotlib`. 

> **Nota de rendimiento:** Separar la generación gráfica del guardado en disco tiene una ventaja crítica en `Databricks`. Permite al orquestador llamar a `plt.close("all")` inmediatamente después de guardar la imagen, liberando la memoria en el clúster antes de renderizar el siguiente gráfico, previniendo fugas de memoria.


### 4.1. Curvas de rendimiento global (*PR* y *ROC*)

Estas curvas evalúan el modelo en todos los umbrales de decisión posibles:

* **Curva *PR*:** Especialmente útil para conjuntos de datos desbalanceados (como el fraude), donde la línea base aleatoria depende de la tasa real de casos positivos.
* **Curva *ROC*:** Muestra el equilibrio entre verdaderos positivos y falsos positivos.

In [0]:
def fig_pr_curve(y_true, y_prob, auc_pr, title):
    """
    Plot the PR curve with AUC annotation and a shaded area.
    Preferred over ROC for imbalanced datasets: the random baseline equals
    the positive class rate, not a fixed diagonal.
    """
    precision_values, recall_values, _ = precision_recall_curve(y_true, y_prob)
    baseline = y_true.mean()
    fig, ax = plt.subplots(figsize = fig_size_standard)
    ax.plot(recall_values, precision_values, lw = 2, color = color_pr, label = f"AUC-PR = {auc_pr:.4f}")
    ax.axhline(baseline, color = color_random_baseline, linestyle = '--', lw = 1, label = f"Random baseline = {baseline:.3f}")
    ax.fill_between(recall_values, precision_values, alpha = 0.08, color = color_pr)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha = 0.3)
    plt.tight_layout()
    return fig


def fig_roc_curve(y_true, y_prob, auc_roc, title):
    """
    Plot the ROC curve with AUC annotation and a shaded area under the curve.
    The random classifier diagonal (AUC = 0.5) is shown as a dashed baseline.
    """
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize = fig_size_standard)
    ax.plot(fpr, tpr, lw = 2, color = color_roc, label = f"AUC-ROC = {auc_roc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw = 1, alpha = 0.5, label = "Random classifier")
    ax.fill_between(fpr, tpr, alpha = 0.08, color = color_roc)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(loc = "lower right")
    ax.grid(alpha = 0.3)
    plt.tight_layout()
    return fig


### 4.2. Matriz de confusión normalizada
Muestra los aciertos y errores del modelo bajo un umbral de decisión específico. La generamos en formato dual (conteos crudos y porcentajes normalizados por fila) para revelar la tasa real de detección por clase independientemente del desbalance de los datos.

In [0]:
def fig_confusion_matrix(y_true, y_pred, title):
    """
    Plot the confusion matrix as two side-by-side panels: raw counts and
    row-normalized percentages. Row normalization reveals the per-class
    detection rate independently of class frequency.
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis = 1, keepdims = True) * 100
    fig, axes = plt.subplots(1, 2, figsize = fig_size_confusion)
    for ax, data, fmt, subtitle in [
        (axes[0], cm, "d", "Counts"),
        (axes[1], cm_pct, ".1f", "Row %")
    ]:
        im = ax.imshow(data, cmap = "Blues")
        plt.colorbar(im, ax = ax)
        for i in range(2):
            for j in range(2):
                ax.text(
                    j, i,
                    format(data[i, j], fmt),
                    ha = "center", va = "center",
                    color = "white" if data[i, j] > data.max() / 2 else "black",
                    fontsize = 12, fontweight = "bold"
                )
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Legit", "Fraud"])
        ax.set_yticklabels(["Legit", "Fraud"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"{title} — {subtitle}")
    plt.tight_layout()
    return fig


### 4.3. Interpretación y calibración

* **Coeficientes de regresión logística:** Muestra las variables con mayor impacto en la decisión del modelo (rojo empuja hacia fraude, azul hacia transacción legítima).
* **Curva de calibración:** Evalúa la correspondencia exacta entre las probabilidades que emite el modelo y la tasa real de casos positivos. En sistemas de detección de fraude, la probabilidad se utiliza habitualmente como un *score* continuo de riesgo para priorizar revisiones manuales, por lo que necesita ser matemáticamente honesta. Si el modelo asigna una probabilidad de fraude de 0.8 a un segmento de transacciones, el 80% de ellas deberían ser realmente fraudulentas. Un modelo perfectamente calibrado se alineará con la diagonal principal: desviaciones por debajo indican sobreconfianza (asigna probabilidades más altas de lo debido), y por encima, subestimación del riesgo.

In [0]:
def fig_lr_coefficients(coef_array, feature_names, title):
    """
    Plot the top logistic regression coefficients sorted by absolute value. 
    Red means positive coefficient (push toward fraud) and
    blue means negative (push toward legit).
    """
    coef = np.array(coef_array)
    n = min(coefficients_top_n, len(coef))
    idx = np.argsort(np.abs(coef))[-n:][::-1]
    fig_height = max(fig_size_coef_min_h, n * fig_size_coef_row_h)
    fig, ax = plt.subplots(figsize = (fig_size_coef_width, fig_height))
    colors = [color_positive_coef if c > 0 else color_negative_coef for c in coef[idx]]
    ax.barh(range(n), coef[idx], color = colors, edgecolor = "white", linewidth = 0.5)
    ax.set_yticks(range(n))
    ax.set_yticklabels([feature_names[i] for i in idx], fontsize = 8)
    ax.axvline(0, color = "black", lw = 0.8)
    ax.set_xlabel("Coefficient value")
    ax.set_title(title)
    ax.legend(
        handles=[
            Patch(color = color_positive_coef, label = "Indicative of fraud"),
            Patch(color = color_negative_coef, label = "Indicative of legit")
        ],
        loc = "lower right",
        fontsize = 8
    )
    ax.invert_yaxis()
    plt.tight_layout()
    return fig


def fig_calibration_curve(y_true, y_prob, title):
    """
    Plot the calibration curve (reliability diagram). A perfectly calibrated
    model follows the diagonal. Deviations above it indicate under-confidence;
    below it, over-confidence.
    """
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins = calibration_n_bins)
    fig, ax = plt.subplots(figsize = fig_size_standard)
    ax.plot(mean_pred, frac_pos, "s-", lw = 2, color = color_calibration, label = "Model")
    ax.plot([0, 1], [0, 1], "k--", lw = 1, label = "Perfect calibration")
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha = 0.1, color = color_calibration)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha = 0.3)
    plt.tight_layout()
    return fig


### 4.4. Búsqueda de umbral óptimo (*threshold sweep*)

Grafica la evolución de *precision*, *recall* y *F1-score* para cada punto de decisión. Es vital para identificar visualmente el umbral operativo que maximiza el valor de negocio.

In [0]:
def fig_threshold_sweep(y_true, y_prob, title):
    """
    Plot precision, recall, and F1-score across the full range of decision thresholds.
    The vertical dashed line marks the threshold that maximizes F1-score.
    """
    thresholds = np.linspace(threshold_sweep_start, threshold_sweep_stop, threshold_sweep_steps)
    precisions, recalls, f1s = [], [], []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f = 2 * p * r / (p + r) if (p + r) > 0 else 0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f)
    best_t = thresholds[np.argmax(f1s)]
    fig, ax = plt.subplots(figsize = fig_size_wide)
    ax.plot(thresholds, precisions, label = "Precision", color = color_pr)
    ax.plot(thresholds, recalls, label = "Recall", color = color_roc)
    ax.plot(thresholds, f1s, label = "F1", color = color_calibration, lw = 2)
    ax.axvline(best_t, color = color_random_baseline, linestyle = "--", lw = 1, label = f"Best F1 threshold = {best_t:.2f}")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha = 0.3)
    plt.tight_layout()
    return fig


## 5. Construcción del *pipeline* de preprocesado

La función `build_preprocessing_stages` recibe todos los hiperparámetros de preprocesado como **parámetros explícitos** de la función, en lugar de leerlos pasivamente desde variables globales del entorno.

Este enfoque arquitectónico ofrece dos grandes ventajas:

1. **Trazabilidad estricta**: Queda documentado y explícito en la firma de la función qué valores controlan el comportamiento de cada etapa, sin necesidad de buscar dónde se definieron las variables en otras libretas.
2. **Seguridad frente a mutaciones**: Si `07_Utils.py` modificara accidentalmente alguna variable entre llamadas durante un flujo complejo, los hiperparámetros de nuestro entrenamiento seguirían estando protegidos al ser inyectados por valor.

> **Nota sobre variables estructurales:** Las variables que definen la estructura inmutable del *pipeline* (como `imputer_input_columns`, las sentencias `SQL` estáticas, etc.) sí continúan leyéndose del espacio de nombres heredado de `07_Utils.py`, ya que son una consecuencia directa y estática del esquema del conjunto de datos, no hiperparámetros sujetos a ajuste u optimización.

In [0]:
def build_preprocessing_stages(
    imputer_strategy,
    var_selector_threshold,
    scaler_with_mean,
    scaler_with_std,
    ohe_drop_last,
    si_handle_invalid,
    si_order_type,
    ohe_handle_invalid,
    asm_handle_invalid
):
    """
    Return a fresh list of preprocessing stages for one pipeline run.

    Creates new `Estimator` instances on every call so that the fitted state
    (vocabularies, medians, scales, etc.) from a previous `Pipeline.fit()`
    never leaks into the next run.

    Structural column lists and `SQL` statements are read from the `07_Utils.py`
    namespace. Hyperparameter values come exclusively from the function parameters.
    """
    # 1. Drop high-cardinality identifiers and redundant columns
    drop_transformer = SQLTransformer(statement = drop_statement)

    # 2. Imputation for nullable numeric columns
    imputer = Imputer(
        inputCols = imputer_input_columns,
        outputCols = imputer_output_columns,
        strategy = imputer_strategy
    )

    # 3. Boolean flags cast to DOUBLE with inline null imputation via COALESCE
    boolean_transformer = SQLTransformer(statement = boolean_statement)

    # 4. Feature engineering from current-transaction fields only
    feature_engineer = SQLTransformer(statement = feature_engineering_statement)

    # 5. Learn category vocabularies
    string_indexer = StringIndexer(
        inputCols = string_indexer_input_columns,
        outputCols = string_indexer_output_columns,
        handleInvalid = si_handle_invalid,
        stringOrderType = si_order_type
    )

    # 6. One-hot encoding
    ohe = OneHotEncoder(
        inputCols = ohe_input_columns,
        outputCols = ohe_output_columns,
        handleInvalid = ohe_handle_invalid,
        dropLast = ohe_drop_last
    )

    # 7. Assemble all feature columns into a single dense or sparse vector
    assembler = VectorAssembler(
        inputCols = assembler_input_columns,
        outputCol = assembler_output_column,
        handleInvalid = asm_handle_invalid
    )

    # 8. Remove quasi-constant features before scaling
    var_selector = VarianceThresholdSelector(
        featuresCol = var_selector_input_column,
        outputCol = var_selector_output_column,
        varianceThreshold = var_selector_threshold
    )

    # 9. Normalize to unit variance, preserving sparsity
    standard_scaler = StandardScaler(
        inputCol = scaler_input_column,
        outputCol = scaler_output_column,
        withMean = scaler_with_mean,
        withStd = scaler_with_std
    )

    preprocessing_stages = [
        drop_transformer,
        imputer,
        boolean_transformer,
        feature_engineer,
        string_indexer,
        ohe,
        assembler,
        var_selector,
        standard_scaler
    ]

    return preprocessing_stages

In [0]:
# Sanity check: list stage types to confirm the pipeline is wired correctly
_stages = build_preprocessing_stages(
    imputer_strategy = imputer_strategy,
    var_selector_threshold = var_selector_threshold,
    scaler_with_mean = scaler_with_mean,
    scaler_with_std = scaler_with_std,
    ohe_drop_last = ohe_drop_last,
    si_handle_invalid = si_handle_invalid,
    si_order_type = si_order_type,
    ohe_handle_invalid = ohe_handle_invalid,
    asm_handle_invalid = asm_handle_invalid
)

for i, stage in enumerate(_stages, 1):
    print(f"{i}. {type(stage).__name__}")


## 6. Construcción y entrenamiento del *pipeline* completo

En esta fase, juntamos las transformaciones y el algoritmo final en un único flujo de trabajo y procedemos a su entrenamiento. Creamos instancias **completamente nuevas** de todas las etapas de preprocesado llamando a nuestra función `build_preprocessing_stages` e inyectándole los hiperparámetros recibidos por los *widgets*. Esto garantiza matemáticamente que el estado ajustado de una ejecución anterior (vocabularios, varianzas, medias, etc.) no contamine la iteración actual.

In [0]:
preprocessing_stages = build_preprocessing_stages(
    imputer_strategy = imputer_strategy,
    var_selector_threshold = var_selector_threshold,
    scaler_with_mean = scaler_with_mean,
    scaler_with_std = scaler_with_std,
    ohe_drop_last = ohe_drop_last,
    si_handle_invalid = si_handle_invalid,
    si_order_type = si_order_type,
    ohe_handle_invalid = ohe_handle_invalid,
    asm_handle_invalid = asm_handle_invalid
)

lr_clf = LogisticRegression(
    featuresCol = features_column,
    labelCol = label_column,
    weightCol = class_weight_column,
    maxIter = max_iter,
    regParam = reg_param,
    elasticNetParam = elastic_net_param,
    family = family,
    standardization = standardization,
    threshold = threshold
)

pipeline_stages = preprocessing_stages + [lr_clf]
full_pipeline = Pipeline(stages = pipeline_stages)
pipeline_model = full_pipeline.fit(train_weighted)
lr_fitted = pipeline_model.stages[-1]

print("Pipeline fitted successfully.")
print(f"Total iterations: {lr_fitted.summary.totalIterations}")


## 7. Cálculo de métricas de entrenamiento y validación

En esta fase, calculamos las seis métricas definidas anteriormente tanto para el conjunto de entrenamiento como para el de validación. Analizar estos resultados en paralelo nos permite evaluar dos aspectos críticos del modelo:

* **La brecha de generalización (`gap_auc_pr`)**: Se obtiene al calcular la diferencia entre el *AUC-PR* de entrenamiento y el de validación. Un valor positivo grande es un indicador directo de **sobreajuste (*overfitting*)**, señalando que el modelo está memorizando los datos en lugar de aprender patrones generalizables.
* **Métrica de selección (*AUC-PR*)**: Utilizamos el *AUC-PR* como nuestro criterio principal para elegir el mejor modelo del *grid*. A diferencia de la curva *ROC* o el *accuracy*, el *AUC-PR* no se ve inflado artificialmente por la abrumadora cantidad de verdaderos negativos (transacciones legítimas), lo que la convierte en la métrica más robusta y honesta para problemas de detección de fraude con un fuerte desbalance de clases.

In [0]:
train_predictions = pipeline_model.transform(train_weighted)
train_metrics = compute_metrics(train_predictions)

validation_predictions = pipeline_model.transform(validation_df)
validation_metrics = compute_metrics(validation_predictions)

gap_auc_pr = train_metrics["auc_pr"] - validation_metrics["auc_pr"]
gap_auc_roc = train_metrics["auc_roc"] - validation_metrics["auc_roc"]

print(f"Train AUC-PR: {train_metrics['auc_pr']:.4f}")
print(f"Validation AUC-PR: {validation_metrics['auc_pr']:.4f}")
print(f"Generalization gap: {gap_auc_pr:+.4f}")


## 8. Conversión a `pandas` y umbral de decisión óptimo

El **umbral óptimo** es el valor de corte sobre la probabilidad (`prob_fraud_column`) que maximiza el *F1-score* en el conjunto de validación. 

Se busca sobre el conjunto de validación, y no sobre el de entrenamiento, para evitar introducir sesgos de sobreajuste. Además, se utiliza exactamente el mismo *grid* de umbrales que la función `fig_threshold_sweep` para garantizar una coherencia total con la figura que generaremos posteriormente.

In [0]:
train_pandas_df = to_pandas_predictions(train_predictions)
validation_pandas_df = to_pandas_predictions(validation_predictions)

y_train = train_pandas_df[label_column].values
p_train = train_pandas_df[prob_fraud_column].values
pred_train = train_pandas_df[prediction_column].values

y_validation = validation_pandas_df[label_column].values
p_validation = validation_pandas_df[prob_fraud_column].values
pred_validation = validation_pandas_df[prediction_column].values

thresholds = np.linspace(threshold_sweep_start, threshold_sweep_stop, threshold_sweep_steps)
best_f1_score = 0.0
best_threshold_val = 0.5

for t in thresholds:
    y_pred_at_t = (p_validation >= t).astype(int)
    tp = ((y_pred_at_t == 1) & (y_validation == 1)).sum()
    fp = ((y_pred_at_t == 1) & (y_validation == 0)).sum()
    fn = ((y_pred_at_t == 0) & (y_validation == 1)).sum()

    precision_at_t = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_at_t = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_at_t = (
        2 * precision_at_t * recall_at_t / (precision_at_t + recall_at_t)
        if (precision_at_t + recall_at_t) > 0 else 0.0
    )

    if f1_at_t > best_f1_score:
        best_f1_score = f1_at_t
        best_threshold_val = t

print(f"Best threshold (validation): {best_threshold_val:.2f}")
print(f"Best F1-score at threshold: {best_f1_score:.4f}")


## 9. Generación y guardado de figuras

En esta etapa, se generan las diez figuras de diagnóstico y se guardan como archivos `.png` físicos dentro del volumen de `Unity Catalog`.

Antes de generar las figuras, se calculan los nombres reales de las características seleccionadas. El `VarianceThresholdSelector` elimina las características de baja varianza del vector ensamblado, por lo que el número de coeficientes del modelo es menor que el número de columnas de entrada del `VectorAssembler`. Para alinear correctamente cada coeficiente con su característica, se transforma una sola fila con el *pipeline* ajustado para recuperar los metadatos del vector ensamblado, y se cruzan con los índices seleccionados por el `VarianceThresholdSelector`.

> **Nota metodológica**: Las figuras de calibración y el barrido de umbral (*threshold sweep*) se calculan **exclusivamente sobre el conjunto de validación**. Evaluarlas y visualizarlas sobre los datos de entrenamiento ofrecería una visión sesgada y poco representativa del comportamiento real del modelo ante datos nuevos.

In [0]:
# Create the run directory in the volume
run_tmp_path = str(Path(uc_volume_path) / "runs" / run_tag)
figures_local_path = str(Path(run_tmp_path) / "figures")
dbutils.fs.mkdirs(figures_local_path)

lr_coefficients = lr_fitted.coefficients.toArray().tolist()

In [0]:
# Transform a single row to retrieve the expanded feature names from the
# assembler output metadata. This is necessary because one-hot encoded
# columns are vectors that expand into multiple dimensions at runtime,
# so the exact feature names and count are only known after fitting.
attrs = (
    pipeline_model
    .transform(train_weighted.limit(1))
    .schema["features"]
    .metadata["ml_attr"]["attrs"]
)

# Collect all attribute types sorted by their index to preserve the original order
all_attrs = (
    attrs.get("numeric", [])
    + attrs.get("binary", [])
    + attrs.get("nominal", [])
)
expanded_feature_names = [
    attr["name"]
    for attr in sorted(all_attrs, key = lambda x: x["idx"])
]

# Use the selected features indices from the fitted selector
# to keep only the features that survived the variance filter.
var_selector_fitted = next(
    stage for stage in pipeline_model.stages
    if isinstance(stage, VarianceThresholdSelectorModel)
)
selected_feature_names = [
    expanded_feature_names[i]
    for i in var_selector_fitted.selectedFeatures
]

print(f"Assembler input columns ({len(expanded_feature_names)}): {expanded_feature_names}")
print(f"Selected features ({len(selected_feature_names)}): {selected_feature_names}")
print(f"Coefficients ({len(lr_coefficients)}): {lr_coefficients}")

In [0]:
def _save_fig(fig, filename):
    """Save a figure to the run temporal directory and close it to free memory."""
    fig.savefig(Path(figures_local_path) / filename, dpi = 150, bbox_inches = "tight")
    plt.close("all")


_save_fig(
    fig_pr_curve(y_train, p_train, train_metrics["auc_pr"], f"PR — {run_tag} (train)"),
    "train_pr_curve.png"
)
_save_fig(
    fig_pr_curve(y_validation, p_validation, validation_metrics["auc_pr"], f"PR — {run_tag} (validation)"),
    "val_pr_curve.png"
)

_save_fig(
    fig_roc_curve(y_train, p_train, train_metrics["auc_roc"], f"ROC — {run_tag} (train)"),
    "train_roc_curve.png"
)
_save_fig(
    fig_roc_curve(y_validation, p_validation, validation_metrics["auc_roc"], f"ROC - {run_tag} (validation)"),
    "val_roc_curve.png"
)

_save_fig(
    fig_confusion_matrix(y_train, pred_train, f"Confusion matrix — {run_tag} (train)"),
    "train_confusion_matrix.png"
)
_save_fig(
    fig_confusion_matrix(y_validation, pred_validation, f"Confusion matrix — {run_tag} (validation)"),
    "val_confusion_matrix.png"
)

_save_fig(
    fig_lr_coefficients(lr_coefficients, selected_feature_names, f"Coefficients — {run_tag}"),
    "lr_coefficients.png"
)

_save_fig(
    fig_calibration_curve(y_train, p_train, f"Calibration (train) — {run_tag}"),
    "train_calibration.png"
)
_save_fig(
    fig_calibration_curve(y_validation, p_validation, f"Calibration (validation) — {run_tag}"),
    "val_calibration.png"
)

_save_fig(
    fig_threshold_sweep(y_validation, p_validation, f"Threshold sweep (validation) — {run_tag}"),
    "val_threshold_sweep.png"
)

print("All diagnostic figures generated and saved successfully.")


## 10. Serialización del *pipeline* entrenado

El `PipelineModel` completo se escribe físicamente en el volumen de `Unity Catalog` en formato nativo de `Spark ML`.

In [0]:
model_save_path = str(Path(run_tmp_path) / "pipeline_model")
dbutils.fs.mkdirs(model_save_path)
pipeline_model.write().overwrite().save(model_save_path)
print(f"Pipeline model successfully saved to {model_save_path}")


## 11. Coeficientes como `.csv`

Se exportan los coeficientes ordenados por valor absoluto. Este artefacto complementa la figura de barras con un registro tabular que facilita la interpretabilidad del modelo.

In [0]:
coef_df = pd.DataFrame({
    "feature": selected_feature_names,
    "coefficient": lr_coefficients
})

coef_df_sorted = (
    coef_df
    .assign(abs_coef = lambda df: df["coefficient"].abs())
    .sort_values(by = "abs_coef", ascending = False)
    .drop(columns = ["abs_coef"])
)

coefficients_csv_path = str(Path(run_tmp_path) / "lr_coefficients.csv")
coef_df_sorted.to_csv(coefficients_csv_path, index = False)

print(f"Logistic regression coefficients successfully saved to {coefficients_csv_path}")


## 12. Ejemplos de entrada y salida para la firma del modelo

Se guardan como archivos `parquet` en el volumen de `Unity Catalog` para inferir correctamente el esquema de entrada y salida (la "firma") del modelo antes de registrarlo.

In [0]:
signature_sample_size = 5
transform_buffer_size = 300

input_example_pandas_df = train_weighted.limit(signature_sample_size).toPandas()

sample_predictions = pipeline_model.transform(train_weighted.limit(transform_buffer_size))
output_example_pandas_df = (
    to_pandas_predictions(sample_predictions)[[prob_fraud_column, prediction_column]].head(signature_sample_size)
)

# Clean up residual Spark Connect metadata to avoid serialization issues
clean_input_df = pd.DataFrame(input_example_pandas_df.to_dict("list"))
clean_output_df = pd.DataFrame(output_example_pandas_df.to_dict("list"))

input_example_path = str(Path(run_tmp_path) / "input_example.parquet")
output_example_path = str(Path(run_tmp_path) / "output_example.parquet")

clean_input_df.to_parquet(input_example_path, index = False)
clean_output_df.to_parquet(output_example_path, index = False)

print(f"Input examples successfully saved to {input_example_path}")
print(f"Output examples successfully saved to {output_example_path}")


## 13. Informes de clasificación

La función `classification_report` genera una tabla detallada con *precision*, *recall* y *F1-score* desglosados por clase.

Estos informes se guardan como ficheros de texto plano (`.txt`) en el volumen de `Unity Catalog`.

In [0]:
target_names = ["Legit", "Fraud"]

train_report_path = str(Path(run_tmp_path) / "train_classification_report.txt")
validation_report_path = str(Path(run_tmp_path) / "val_classification_report.txt")

with open(train_report_path, "w") as fh:
    fh.write(classification_report(y_train, pred_train, target_names = target_names))

with open(validation_report_path, "w") as fh:
    fh.write(classification_report(y_validation, pred_validation, target_names = target_names))

print(f"Classification training report successfully saved to {train_report_path}")
print(f"Classification validation report successfully saved to {validation_report_path}")


## 14. Metadatos de convergencia del optimizador

El atributo `objectiveHistory` contiene el valor de la función de pérdida (*loss function*) calculado al final de cada iteración por el optimizador matemático (habitualmente `L-BFGS`).

Extraemos este historial como una lista nativa de `Python` para auditar si el modelo convergió de forma natural o si se detuvo prematuramente por alcanzar el límite máximo de iteraciones (`max_iter`).

In [0]:
has_history = (
    hasattr(lr_fitted, "summary")
    and hasattr(lr_fitted.summary, "objectiveHistory")
)

objective_history = list(lr_fitted.summary.objectiveHistory) if has_history else []
total_iterations = int(lr_fitted.summary.totalIterations) if has_history else max_iter

convergence_metadata = {
    "objective_history": objective_history,
    "total_iterations": total_iterations,
    "converged": float(len(objective_history) < max_iter) if has_history else 0.0,
    "lr_intercept": float(lr_fitted.intercept)
}

print(f"Initial loss: {objective_history[0]:.6f}")
print(f"Final loss: {objective_history[-1]:.6f}")
print(f"Converged: {bool(convergence_metadata['converged'])}")


## 15. Liberación de memoria y retorno del resultado

Como paso final, eliminamos explícitamente los objetos de `Spark` de mayor tamaño en memoria (`DataFrames` de predicciones, el *pipeline* y el modelo) y forzamos la recolección de basura nativa de `Python` (`gc.collect()`). Esta práctica defensiva reduce drásticamente el riesgo de que la caché del clúster se sature antes de que termine la sesión.

Finalmente, utilizamos `dbutils.notebook.exit` para devolver el control a la libreta orquestadora. Puesto que esta función solo admite cadenas de texto, empaquetamos todos los hiperparámetros, rutas de artefactos y métricas en un diccionario y lo serializamos a formato `.json`.

In [0]:
del train_predictions, validation_predictions
del pipeline_model, full_pipeline, lr_clf, preprocessing_stages, lr_fitted
gc.collect()

result = {
    "run_tag": run_tag,
    "reg_param": reg_param,
    "elastic_net": elastic_net_param,
    "max_iter": max_iter,
    "family": family,
    "standardization": standardization,
    "threshold": threshold,
    "imputer_strategy": imputer_strategy,
    "var_selector_threshold": var_selector_threshold,
    "scaler_with_mean": scaler_with_mean,
    "scaler_with_std": scaler_with_std,
    "ohe_drop_last": ohe_drop_last,
    "si_handle_invalid": si_handle_invalid,
    "si_order_type": si_order_type,
    "ohe_handle_invalid": ohe_handle_invalid,
    "asm_handle_invalid": asm_handle_invalid,
    "model_save_path": model_save_path,
    "input_example_path": input_example_path,
    "output_example_path": output_example_path,
    "figures_local_path": figures_local_path,
    "coefficients_csv_path": coefficients_csv_path,
    "train_report_path": train_report_path,
    "validation_report_path": validation_report_path,
    "convergence": convergence_metadata,
    "lr_coefficients": lr_coefficients,
    "train_metrics": train_metrics,
    "validation_metrics": validation_metrics,
    "gap_auc_pr": gap_auc_pr,
    "gap_auc_roc": gap_auc_roc,
    "best_threshold": best_threshold_val,
    "best_f1_at_threshold": best_f1_score
}

print(f"Exiting notebook and returning results for run: {run_tag}")
dbutils.notebook.exit(json.dumps(result))